<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/09)_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 현재 src 파일 구조 확인


for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
decoder_layer.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# 기존 decoderlayer 확인

print(
    (SRC_DIR / "decoder_layer.py").read_text()
)


import torch.nn as nn

from src.multi_head_attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1,
    ):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads,
        )

        self.feed_forward = PositionwiseFeedForward(
            d_model,
            d_ff,
        )

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(
        self,
        x,
        encoder_output,
        self_attention_mask=None,
        cross_atten

In [ ]:
# 기존 Encoder 구조 확인

print(
    (SRC_DIR / "encoder.py").read_text()
)


import torch.nn as nn

from src.encoder_layer import EncoderLayer


class Encoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        attention_weights_list = []

        for layer in self.layers:
            x, attention_weights = layer(
                x,
                mask,
            )

            attention_weights_list.append(
                attention_weights
            )

        return x, attention_weights_list



In [ ]:
# src/decoder.py 전체 코드

import torch.nn as nn

from src.decoder_layer import DecoderLayer


class Decoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        x,
        encoder_output,
        self_attention_mask=None,
        cross_attention_mask=None,
    ):
        self_attention_weights_list = []
        cross_attention_weights_list = []

        for layer in self.layers:
            (
                x,
                self_attention_weights,
                cross_attention_weights,
            ) = layer(
                x,
                encoder_output,
                self_attention_mask,
                cross_attention_mask,
            )

            self_attention_weights_list.append(
                self_attention_weights
            )

            cross_attention_weights_list.append(
                cross_attention_weights
            )

        return (
            x,
            self_attention_weights_list,
            cross_attention_weights_list,
        )

In [ ]:
# 파일로 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/decoder.py

import torch.nn as nn

from src.decoder_layer import DecoderLayer


class Decoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        x,
        encoder_output,
        self_attention_mask=None,
        cross_attention_mask=None,
    ):
        self_attention_weights_list = []
        cross_attention_weights_list = []

        for layer in self.layers:
            (
                x,
                self_attention_weights,
                cross_attention_weights,
            ) = layer(
                x,
                encoder_output,
                self_attention_mask,
                cross_attention_mask,
            )

            self_attention_weights_list.append(
                self_attention_weights
            )

            cross_attention_weights_list.append(
                cross_attention_weights
            )

        return (
            x,
            self_attention_weights_list,
            cross_attention_weights_list,
        )

Writing /content/drive/MyDrive/attention_is_all_you_need/src/decoder.py


In [ ]:
# Decoder import

import torch

from src.decoder import Decoder

In [ ]:
# 기본 설정

batch_size = 2

source_len = 5
target_len = 4

d_model = 8
num_heads = 2
d_ff = 32

num_layers = 3

dropout = 0.0

In [ ]:
# Decoder 생성

decoder = Decoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=dropout,
)

print(decoder)

Decoder(
  (layers): ModuleList(
    (0-2): 3 x DecoderLayer(
      (self_attention): MultiHeadAttention(
        (W_Q): Linear(in_features=8, out_features=8, bias=True)
        (W_K): Linear(in_features=8, out_features=8, bias=True)
        (W_V): Linear(in_features=8, out_features=8, bias=True)
        (attention): ScaledDotProductAttention()
        (W_O): Linear(in_features=8, out_features=8, bias=True)
      )
      (cross_attention): MultiHeadAttention(
        (W_Q): Linear(in_features=8, out_features=8, bias=True)
        (W_K): Linear(in_features=8, out_features=8, bias=True)
        (W_V): Linear(in_features=8, out_features=8, bias=True)
        (attention): ScaledDotProductAttention()
        (W_O): Linear(in_features=8, out_features=8, bias=True)
      )
      (feed_forward): PositionwiseFeedForward(
        (linear1): Linear(in_features=8, out_features=32, bias=True)
        (linear2): Linear(in_features=32, out_features=8, bias=True)
      )
      (dropout1): Dropout(p=0.

In [ ]:
# Layer 개수 확인

print(
    "Number of DecoderLayers:",
    len(decoder.layers),

)
assert len(decoder.layers) == num_layers

Number of DecoderLayers: 3


In [ ]:
# DecoderLayer 객체 독립성 확인

print(
    decoder.layers[0]
    is decoder.layers[1]
)

# 확실하게 확인

assert (
    decoder.layers[0]
    is not decoder.layers[1]
)

assert (
    decoder.layers[1]
    is not decoder.layers[2]
)

print("DecoderLayer objects are independent.")

False
DecoderLayer objects are independent.


In [ ]:
# LayerParameter 독립성 확인

layer0_parameter = next(
    decoder.layers[0].parameters()
)

layer1_parameter = next(
    decoder.layers[1].parameters()
)

layer2_parameter = next(
    decoder.layers[2].parameters()
)

print(
    layer0_parameter
    is layer1_parameter
)

print(
    layer1_parameter
    is layer2_parameter
)

assert (
    layer0_parameter
    is not layer1_parameter
)

assert (
    layer1_parameter
    is not layer2_parameter
)

print("DecoderLayer parameters are not shared.")

False
False
DecoderLayer parameters are not shared.


In [ ]:
# 랜덤 Decoder input 생성

x = torch.randn(
    batch_size,
    target_len,
    d_model,
)

print("Decoder input shape:", x.shape)

Decoder input shape: torch.Size([2, 4, 8])


In [ ]:
# 랜덤 Encoder output 생성

encoder_output = torch.randn(
    batch_size,
    source_len,
    d_model,
)

print(
    "Encoder output shape:",
    encoder_output.shape,
)

Encoder output shape: torch.Size([2, 5, 8])


In [ ]:
# Mask 없이 forward

(
    output,
    self_attention_weights_list,
    cross_attention_weights_list,
) = decoder(
    x,
    encoder_output,
)

In [ ]:
# 최종 Decoder output Shape 확인

print(
    "Decoder output shape:",
    output.shape,
)

assert output.shape == (
    batch_size,
    target_len,
    d_model,
)

Decoder output shape: torch.Size([2, 4, 8])


In [ ]:
# Attention weight list 길이

print(
    "Self-Attention layers:",
    len(self_attention_weights_list),
)

print(
    "Cross-Attention layers:",
    len(cross_attention_weights_list),
)

assert (
    len(self_attention_weights_list)
    == num_layers
)

assert (
    len(cross_attention_weights_list)
    == num_layers
)

Self-Attention layers: 3
Cross-Attention layers: 3


In [ ]:
# 모든 Layer의 Attention Shape 확인

for i in range(num_layers):
    print(
        f"Layer {i + 1} "
        f"Self-Attention:",
        self_attention_weights_list[i].shape,
    )

    print(
        f"Layer {i + 1} "
        f"Cross-Attention:",
        cross_attention_weights_list[i].shape,
    )


# 검증

for self_weights in self_attention_weights_list:
    assert self_weights.shape == (
        batch_size,
        num_heads,
        target_len,
        target_len,
    )

for cross_weights in cross_attention_weights_list:
    assert cross_weights.shape == (
        batch_size,
        num_heads,
        target_len,
        source_len,
    )

print("All attention shapes are correct.")

Layer 1 Self-Attention: torch.Size([2, 2, 4, 4])
Layer 1 Cross-Attention: torch.Size([2, 2, 4, 5])
Layer 2 Self-Attention: torch.Size([2, 2, 4, 4])
Layer 2 Cross-Attention: torch.Size([2, 2, 4, 5])
Layer 3 Self-Attention: torch.Size([2, 2, 4, 4])
Layer 3 Cross-Attention: torch.Size([2, 2, 4, 5])
All attention shapes are correct.


In [ ]:
# Mask 함수 import

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

In [ ]:
# Target token IDs

target_token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

print(target_token_ids)

tensor([[5, 8, 3, 9],
        [7, 2, 0, 0]])


In [ ]:
# Source tokens IDs

source_token_ids = torch.tensor([
    [10, 11, 12, 13, 14],
    [20, 21, 22, 0, 0],
])

print(source_token_ids)

tensor([[10, 11, 12, 13, 14],
        [20, 21, 22,  0,  0]])


In [ ]:
# Target Padding Mask 생성

pad_idx = 0

target_padding_mask = create_padding_mask(
    target_token_ids,
    pad_idx,
)

print(
    "Target padding mask shape:",
    target_padding_mask.shape,
)

Target padding mask shape: torch.Size([2, 1, 1, 4])


In [ ]:
# Casual Mask 생성

causal_mask = create_causal_mask(
    target_len,
)

print(
    "Causal mask shape:",
    causal_mask.shape,
)

Causal mask shape: torch.Size([1, 1, 4, 4])


In [ ]:
# Decoder Self Attention Mask 결합

self_attention_mask = (
    target_padding_mask
    & causal_mask
)

print(
    "Self-Attention mask shape:",
    self_attention_mask.shape,
)

Self-Attention mask shape: torch.Size([2, 1, 4, 4])


In [ ]:
# Source Padding Mask 생성

source_padding_mask = create_padding_mask(
    source_token_ids,
    pad_idx,
)

print(
    "Source padding mask shape:",
    source_padding_mask.shape,
)

Source padding mask shape: torch.Size([2, 1, 1, 5])


In [ ]:
# Mask를 Decoder에 전달

(
    masked_output,
    masked_self_attention_weights_list,
    masked_cross_attention_weights_list,
) = decoder(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

In [ ]:
# Mask 적용 후 output shape 확인

print(
    "Masked Decoder output shape:",
    masked_output.shape,
)

assert masked_output.shape == (
    batch_size,
    target_len,
    d_model,
)

Masked Decoder output shape: torch.Size([2, 4, 8])


In [ ]:
# 모든 Layer에서 미래 Target Attention 확인

future_positions = torch.triu(
    torch.ones(
        target_len,
        target_len,
        dtype=torch.bool,
    ),
    diagonal=1,
)

for layer_idx, weights in enumerate(
    masked_self_attention_weights_list
):
    future_weights = weights[
        :,
        :,
        future_positions,
    ]

    assert torch.all(
        future_weights == 0
    )

    print(
        f"Layer {layer_idx + 1}: "
        "future attention blocked"
    )

Layer 1: future attention blocked
Layer 2: future attention blocked
Layer 3: future attention blocked


In [ ]:
# 모든 Layer에서 Target PAD key

# 두번째 batch의 target = [7, 2, PAD, PAD] -> key index 2 3은 어떤 query에서도 Attention을 받으면 안된다

for layer_idx, weights in enumerate(
    masked_self_attention_weights_list
):
    batch_1_weights = weights[1]

    assert torch.all(
        batch_1_weights[:, :, 2] == 0
    )

    assert torch.all(
        batch_1_weights[:, :, 3] == 0
    )

    print(
        f"Layer {layer_idx + 1}: "
        "target PAD keys blocked"
    )

Layer 1: target PAD keys blocked
Layer 2: target PAD keys blocked
Layer 3: target PAD keys blocked


In [ ]:
# 모든 Layer에서 Source PAD Cross-Attention 확인

# 두번째 source sequence: [20, 21, 22, PAD, PAD] -> source key index 3 4가 모두 0이여야한다


for layer_idx, weights in enumerate(
    masked_cross_attention_weights_list
):
    batch_1_weights = weights[1]

    assert torch.all(
        batch_1_weights[:, :, 3] == 0
    )

    assert torch.all(
        batch_1_weights[:, :, 4] == 0
    )

    print(
        f"Layer {layer_idx + 1}: "
        "source PAD keys blocked"
    )

Layer 1: source PAD keys blocked
Layer 2: source PAD keys blocked
Layer 3: source PAD keys blocked


In [ ]:
# Mask 전달 구조 이해하기

for layer in self.layers:
    ...
    = layer(
        x,
        encoder_output,
        self_attention_mask,
        cross_attention_mask,
    )

SyntaxError: invalid syntax (660729224.py, line 5)

In [ ]:
# batch / sequence length 변화 테스트

# 특정 b, t, s에 종속되지 않았는지 검사

test_shapes = [
    (1, 3, 5),
    (2, 4, 6),
    (3, 5, 4),
]

for B, T, S in test_shapes:
    test_x = torch.randn(
        B,
        T,
        d_model,
    )

    test_encoder_output = torch.randn(
        B,
        S,
        d_model,
    )

    (
        test_output,
        test_self_weights,
        test_cross_weights,
    ) = decoder(
        test_x,
        test_encoder_output,
    )

    assert test_output.shape == (
        B,
        T,
        d_model,
    )

    for weights in test_self_weights:
        assert weights.shape == (
            B,
            num_heads,
            T,
            T,
        )

    for weights in test_cross_weights:
        assert weights.shape == (
            B,
            num_heads,
            T,
            S,
        )

    print(
        f"B={B}, T={T}, S={S}: PASS"
    )

B=1, T=3, S=5: PASS
B=2, T=4, S=6: PASS
B=3, T=5, S=4: PASS


In [ ]:
# num_layers 변화 테스트

for test_num_layers in [
    1,
    2,
    4,
]:
    test_decoder = Decoder(
        d_model=d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        num_layers=test_num_layers,
        dropout=0.0,
    )

    test_x = torch.randn(
        batch_size,
        target_len,
        d_model,
    )

    test_encoder_output = torch.randn(
        batch_size,
        source_len,
        d_model,
    )

    (
        test_output,
        test_self_weights,
        test_cross_weights,
    ) = test_decoder(
        test_x,
        test_encoder_output,
    )

    assert (
        len(test_decoder.layers)
        == test_num_layers
    )

    assert (
        len(test_self_weights)
        == test_num_layers
    )

    assert (
        len(test_cross_weights)
        == test_num_layers
    )

    assert test_output.shape == (
        batch_size,
        target_len,
        d_model,
    )

    print(
        f"num_layers={test_num_layers}: PASS"
    )

num_layers=1: PASS
num_layers=2: PASS
num_layers=4: PASS


In [ ]:
# dropout 0.0 반복 실행 테스트

# 현재 dropout = 0.0이기에 동일한 입력을 여러번 넣어도 동일한 결과가 나와야한다

(
    output1,
    self_weights1,
    cross_weights1,
) = decoder(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

(
    output2,
    self_weights2,
    cross_weights2,
) = decoder(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

# 최종 output 비교

print(
    torch.allclose(
        output1,
        output2,
    )
)

# Attention weights 검사

for w1, w2 in zip(
    self_weights1,
    self_weights2,
):
    assert torch.allclose(
        w1,
        w2,
    )

for w1, w2 in zip(
    cross_weights1,
    cross_weights2,
):
    assert torch.allclose(
        w1,
        w2,
    )

assert torch.allclose(
    output1,
    output2,
)

print(
    "dropout=0.0 deterministic test: PASS"
)

True
dropout=0.0 deterministic test: PASS


In [ ]:
# 최종 통합 테스트

import torch

from src.decoder import Decoder
from src.mask import (
    create_padding_mask,
    create_causal_mask,
)


# -------------------------
# 기본 설정
# -------------------------

batch_size = 2

source_len = 5
target_len = 4

d_model = 8
num_heads = 2
d_ff = 32

num_layers = 3

dropout = 0.0

pad_idx = 0


# -------------------------
# Decoder 생성
# -------------------------

decoder = Decoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=dropout,
)


# -------------------------
# Layer 개수 / 독립성
# -------------------------

assert len(decoder.layers) == num_layers

assert (
    decoder.layers[0]
    is not decoder.layers[1]
)

assert (
    decoder.layers[1]
    is not decoder.layers[2]
)

layer0_parameter = next(
    decoder.layers[0].parameters()
)

layer1_parameter = next(
    decoder.layers[1].parameters()
)

assert (
    layer0_parameter
    is not layer1_parameter
)


# -------------------------
# 입력 생성
# -------------------------

x = torch.randn(
    batch_size,
    target_len,
    d_model,
)

encoder_output = torch.randn(
    batch_size,
    source_len,
    d_model,
)


# -------------------------
# Mask 없이 실행
# -------------------------

(
    output,
    self_weights_list,
    cross_weights_list,
) = decoder(
    x,
    encoder_output,
)

assert output.shape == (
    batch_size,
    target_len,
    d_model,
)

assert (
    len(self_weights_list)
    == num_layers
)

assert (
    len(cross_weights_list)
    == num_layers
)

for weights in self_weights_list:
    assert weights.shape == (
        batch_size,
        num_heads,
        target_len,
        target_len,
    )

for weights in cross_weights_list:
    assert weights.shape == (
        batch_size,
        num_heads,
        target_len,
        source_len,
    )


# -------------------------
# Mask 생성
# -------------------------

target_token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

source_token_ids = torch.tensor([
    [10, 11, 12, 13, 14],
    [20, 21, 22, 0, 0],
])

target_padding_mask = create_padding_mask(
    target_token_ids,
    pad_idx,
)

causal_mask = create_causal_mask(
    target_len,
)

self_attention_mask = (
    target_padding_mask
    & causal_mask
)

source_padding_mask = create_padding_mask(
    source_token_ids,
    pad_idx,
)


# -------------------------
# Mask 적용 실행
# -------------------------

(
    masked_output,
    masked_self_weights,
    masked_cross_weights,
) = decoder(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

assert masked_output.shape == (
    batch_size,
    target_len,
    d_model,
)


# -------------------------
# 미래 Attention 차단
# -------------------------

future_positions = torch.triu(
    torch.ones(
        target_len,
        target_len,
        dtype=torch.bool,
    ),
    diagonal=1,
)

for weights in masked_self_weights:
    assert torch.all(
        weights[
            :,
            :,
            future_positions,
        ] == 0
    )


# -------------------------
# Target PAD Key 차단
# -------------------------

for weights in masked_self_weights:
    assert torch.all(
        weights[1, :, :, 2] == 0
    )

    assert torch.all(
        weights[1, :, :, 3] == 0
    )


# -------------------------
# Source PAD Key 차단
# -------------------------

for weights in masked_cross_weights:
    assert torch.all(
        weights[1, :, :, 3] == 0
    )

    assert torch.all(
        weights[1, :, :, 4] == 0
    )


# -------------------------
# dropout=0.0 반복 실행
# -------------------------

(
    output1,
    self_weights1,
    cross_weights1,
) = decoder(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

(
    output2,
    self_weights2,
    cross_weights2,
) = decoder(
    x,
    encoder_output,
    self_attention_mask,
    source_padding_mask,
)

assert torch.allclose(
    output1,
    output2,
)

for w1, w2 in zip(
    self_weights1,
    self_weights2,
):
    assert torch.allclose(
        w1,
        w2,
    )

for w1, w2 in zip(
    cross_weights1,
    cross_weights2,
):
    assert torch.allclose(
        w1,
        w2,
    )


print(
    "All Decoder tests passed!"
)

All Decoder tests passed!
